In [ ]:
from pathlib import Path
import sys
import os
import shutil
import yaml
import numpy as np

# =========================
# EXP06 - TOP CONSTANTS
# =========================

# Kaggle source dataset (read-only) and writable workspace copy
SRC_INPUT = Path("/kaggle/input/datasets/ronitraj1/ronit-pm25-phase2-src")
WORK_SRC = Path("/kaggle/working/ronit-pm25-phase2-src")

# Fallback for local/non-kaggle runs
if not SRC_INPUT.exists():
    local_guess = Path.cwd().resolve().parent if (Path.cwd().resolve().parent / "configs").exists() else Path.cwd().resolve()
    SRC_INPUT = local_guess
    WORK_SRC = local_guess

# Create writable copy for config patching and %run execution
if SRC_INPUT != WORK_SRC and SRC_INPUT.exists() and not WORK_SRC.exists():
    print(f"Creating writable workspace copy at {WORK_SRC}")
    shutil.copytree(SRC_INPUT, WORK_SRC)

PROJECT_ROOT = WORK_SRC if WORK_SRC.exists() else SRC_INPUT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure %run scripts/... resolves exactly like exp01 style
os.chdir(PROJECT_ROOT)

# Paths
RAW_ROOT = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/raw"
TEST_INPUT_LOC = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in"
WORK_DIR = Path("/kaggle/working")
EXP_DIR = WORK_DIR / "experiments" / "exp06_hybrid"
EXP_DIR.mkdir(parents=True, exist_ok=True)
LIVE_SYNC_DIR = str(WORK_DIR / "live_checkpoints" / "exp06_hybrid")
LIVE_SYNC_ENABLED = True
LIVE_SYNC_EVERY = 1
LIVE_SYNC_INCLUDE_LOG = True

# Training schedule
EPOCHS = 65
BATCH_SIZE = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4
SCHEDULER_STEP = 10
SCHEDULER_GAMMA = 0.5

# Episode curriculum
EPISODE_ALPHA_WARMUP = 1.0
EPISODE_ALPHA_MAIN = 5.0
EPISODE_ALPHA_FINETUNE = 8.0
EPISODE_MAIN_START_EPOCH = 20
EPISODE_FINETUNE_START_EPOCH = 50
EP_HEAD_WEIGHT = 0.1

# Model constants
MODEL_NAME = "hybrid_ensemble"
WIDTH = 64
MODES1 = 24
MODES2 = 24
CONVLSTM_HIDDEN = 64
INFER_SEASON_IDX = 2  # 0=APRIL,1=JULY,2=OCT,3=DEC

# Runtime toggles
RUN_PREPROCESS = True
RUN_TRAIN = True
RUN_INFER = True

print("SRC_INPUT:", SRC_INPUT)
print("WORK_SRC:", WORK_SRC)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("EXP_DIR:", EXP_DIR)
print("LIVE_SYNC_ENABLED:", LIVE_SYNC_ENABLED)
print("LIVE_SYNC_EVERY:", LIVE_SYNC_EVERY)
print("LIVE_SYNC_DIR:", LIVE_SYNC_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("CWD:", Path.cwd())

In [ ]:
# Preflight checks + build runtime configs from stable base configs
import importlib

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
assert str(PROJECT_ROOT).startswith("/kaggle/working") or not str(SRC_INPUT).startswith("/kaggle/input"), (
    f"Expected writable project root under /kaggle/working, got: {PROJECT_ROOT}"
)
assert (PROJECT_ROOT / "models" / "__init__.py").exists(), f"Missing models package in {PROJECT_ROOT}"
assert (PROJECT_ROOT / "models" / "ensemble.py").exists(), f"Missing ensemble.py in {PROJECT_ROOT / 'models'}"

models_mod = importlib.import_module("models")
print("models imported from:", Path(models_mod.__file__).resolve())
assert str(Path(models_mod.__file__).resolve()).startswith(str(PROJECT_ROOT.resolve())), (
    "models is not imported from PROJECT_ROOT; rerun Cell 1 and ensure CWD/sys.path are correct."
)

train_base = PROJECT_ROOT / "configs" / "train.yaml"
infer_base = PROJECT_ROOT / "configs" / "infer.yaml"

runtime_train_cfg = WORK_DIR / "exp06_train.yaml"
runtime_infer_cfg = WORK_DIR / "exp06_infer.yaml"

with open(train_base, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)

with open(infer_base, "r", encoding="utf-8") as f:
    infer_cfg = yaml.safe_load(f)

stats_path = str(EXP_DIR / "preprocessing" / "normalization_stats.json")

# Train overrides
train_cfg["paths"]["raw_path"] = RAW_ROOT
train_cfg["paths"]["save_dir"] = str(EXP_DIR / "logs" / "log.json")
train_cfg["paths"]["model_save_path"] = str(EXP_DIR / "checkpoints" / "best.pt")
train_cfg["paths"]["live_sync_dir"] = LIVE_SYNC_DIR
train_cfg["training"]["epochs"] = EPOCHS
train_cfg["training"]["batch_size"] = BATCH_SIZE
train_cfg["training"]["lr"] = LR
train_cfg["training"]["weight_decay"] = WEIGHT_DECAY
train_cfg["training"]["scheduler_step"] = SCHEDULER_STEP
train_cfg["training"]["scheduler_gamma"] = SCHEDULER_GAMMA
train_cfg["training"]["live_sync_enabled"] = LIVE_SYNC_ENABLED
train_cfg["training"]["live_sync_every"] = LIVE_SYNC_EVERY
train_cfg["training"]["live_sync_include_log"] = LIVE_SYNC_INCLUDE_LOG
train_cfg["training"]["episode_alpha_warmup"] = EPISODE_ALPHA_WARMUP
train_cfg["training"]["episode_alpha_main"] = EPISODE_ALPHA_MAIN
train_cfg["training"]["episode_alpha_finetune"] = EPISODE_ALPHA_FINETUNE
train_cfg["training"]["episode_main_start_epoch"] = EPISODE_MAIN_START_EPOCH
train_cfg["training"]["episode_finetune_start_epoch"] = EPISODE_FINETUNE_START_EPOCH
train_cfg["training"]["ep_head_weight"] = EP_HEAD_WEIGHT
train_cfg["model"]["name"] = MODEL_NAME
train_cfg["model"]["width"] = WIDTH
train_cfg["model"]["modes1"] = MODES1
train_cfg["model"]["modes2"] = MODES2
train_cfg["model"]["convlstm_hidden"] = CONVLSTM_HIDDEN
train_cfg["preprocessing"] = {"stats_path": stats_path}

# Infer overrides
infer_cfg["paths"]["input_loc"] = TEST_INPUT_LOC
infer_cfg["paths"]["model_path"] = str(EXP_DIR / "checkpoints" / "best.pt")
infer_cfg["paths"]["output_loc"] = str(WORK_DIR / "preds.npy")
infer_cfg["model"]["name"] = MODEL_NAME
infer_cfg["model"]["width"] = WIDTH
infer_cfg["model"]["modes1"] = MODES1
infer_cfg["model"]["modes2"] = MODES2
infer_cfg["model"]["convlstm_hidden"] = CONVLSTM_HIDDEN
infer_cfg["model"]["infer_season_idx"] = INFER_SEASON_IDX
infer_cfg["preprocessing"] = {"stats_path": stats_path}

with open(runtime_train_cfg, "w", encoding="utf-8") as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False)

with open(runtime_infer_cfg, "w", encoding="utf-8") as f:
    yaml.safe_dump(infer_cfg, f, sort_keys=False)

print("Wrote:", runtime_train_cfg)
print("Wrote:", runtime_infer_cfg)
print("Normalization stats path:", stats_path)
print("Train model config:", train_cfg["model"])
print("Live sync settings:", {"enabled": train_cfg["training"].get("live_sync_enabled"), "every": train_cfg["training"].get("live_sync_every"), "include_log": train_cfg["training"].get("live_sync_include_log"), "dir": train_cfg["paths"].get("live_sync_dir")})

In [ ]:
# Preprocess + Train (exp01-style %run pipeline)

if RUN_PREPROCESS:
    print("RUN: scripts/preprocess_data.py")
    %run scripts/preprocess_data.py --config {runtime_train_cfg} --raw_root {RAW_ROOT} --out_dir {EXP_DIR / 'preprocessing'}

if RUN_TRAIN:
    print("RUN: scripts/train.py")
    %run scripts/train.py --config {runtime_train_cfg} --raw_path {RAW_ROOT}

print("Train stage complete.")

In [ ]:
# Inference + submission shape check (exp01-style %run)

if RUN_INFER:
    print("RUN: scripts/infer.py")
    %run scripts/infer.py --config {runtime_infer_cfg} --input_loc {TEST_INPUT_LOC} --model_path {EXP_DIR / 'checkpoints' / 'best.pt'} --output_loc /kaggle/working/preds.npy

pred_path = Path("/kaggle/working/preds.npy")
assert pred_path.exists(), "preds.npy was not created"
preds = np.load(pred_path)
print("preds shape:", preds.shape)
assert preds.shape == (218, 140, 124, 16), f"Invalid submission shape: {preds.shape}"
print("Submission ready:", pred_path)